# Semana 2 · Sesión 2: Inmutabilidad y herencia

**Módulo 0**

## Objetivos de la sesión

1. Explicar por qué los objetos matemáticos deben ser inmutables.
2. Construir jerarquías de clases con herencia y `super()`.
3. Reconocer polimorfismo y *duck typing* en código propio.

## Retomamos donde quedamos

En la sesión 1 construimos `Vector2D` con `__init__`, `@property`, los
métodos especiales de representación y los operadores. La celda de abajo
la trae de vuelta, ya completa, para que este notebook se pueda ejecutar
por su cuenta.

Léela un momento antes de seguir: hoy vamos a atacar justo lo que esta
versión hace **mal**.

In [ ]:
import math


class Vector2D:

    def __init__(self, x, y):
        self.x = x
        self.y = y

    @property
    def magnitud(self):
        return math.sqrt(self.x**2 + self.y**2)

    def __repr__(self):
        return f"Vector2D({self.x}, {self.y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self.x == otro.x and self.y == otro.y

    def __add__(self, otro):
        return Vector2D(self.x + otro.x, self.y + otro.y)

    def __mul__(self, k):
        return Vector2D(self.x * k, self.y * k)

## Mutabilidad vs. inmutabilidad

Un objeto es **mutable** si su estado puede cambiar después de construido.
Nuestro `Vector2D` lo es: nada impide escribir `v.x = 99`.

Eso parece cómodo, y es la fuente de los errores más difíciles de rastrear
en cómputo científico. Si guardas un vector en una lista, en un
diccionario o dentro de otro objeto, y luego alguien lo modifica, el
cambio aparece en todos lados a la vez — porque todos comparten el mismo
objeto, no copias.

In [ ]:
v = Vector2D(3.0, 4.0)
fuerzas = [v, v]   # la misma referencia dos veces, no dos copias

v.x = 99.0         # modificamos "una"...

fuerzas            # ...y cambiaron las dos

In [ ]:
# El mismo objeto v, ahora como llave de un diccionario
try:
    {v: "algo"}
except TypeError as error:
    print("Tampoco puede ser llave de un diccionario:", error)

## Cómo se vuelve inmutable un objeto

El patrón estándar en Python: guardar el dato en un atributo *privado*
(`self._x`, por convención con un guion bajo inicial) y exponerlo con una
`@property` **sin** definir su `setter`. Al no haber setter, asignar
`v.x = 99` lanza `AttributeError`.

Con eso ganamos algo más. Un objeto que no cambia puede definir `__hash__`
de forma segura, y por lo tanto puede usarse como llave de diccionario o
elemento de un conjunto. Fíjate en el detalle: al definir `__eq__` en la
celda anterior, Python **desactivó** el `__hash__` por defecto de la clase,
justo para evitar que un objeto mutable acabe usándose como llave.

**Esta es la regla que siguen todos los objetos matemáticos de SymPy**, y
la razón por la que la vemos hoy: una expresión simbólica nunca se
modifica; las operaciones devuelven expresiones nuevas.

In [ ]:
class Vector2D:

    def __init__(self, x, y):
        self._x = x   # el guion bajo dice "no toques esto desde afuera"
        self._y = y

    @property
    def x(self):
        return self._x   # solo lectura: no definimos un setter

    @property
    def y(self):
        return self._y

    def __repr__(self):
        return f"Vector2D({self._x}, {self._y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self._x == otro._x and self._y == otro._y

    def __hash__(self):
        return hash((self._x, self._y))   # seguro: el estado ya no cambia

    def __add__(self, otro):
        return Vector2D(self._x + otro._x, self._y + otro._y)

In [ ]:
v = Vector2D(3.0, 4.0)

try:
    v.x = 99.0
except AttributeError as error:
    print("No se puede modificar:", error)

{Vector2D(1.0, 0.0): "eje x", Vector2D(0.0, 1.0): "eje y"}   # ahora sí es hashable

## TODO en clase 1

Agrega a la versión inmutable el método especial `__neg__`, que Python
llama cuando escribes `-v`. Debe devolver **un vector nuevo** con ambas
componentes cambiadas de signo.

Es el mismo hábito de siempre, ahora obligatorio: como la clase es
inmutable, no hay forma de "voltear" el vector en su lugar aunque
quisieras. Al terminar, comprueba que `-Vector2D(3.0, 4.0)` da
`Vector2D(-3.0, -4.0)` y que el original no cambió.

In [ ]:
# TODO en clase: agrega __neg__, que devuelve un Vector2D nuevo con el signo cambiado
class Vector2D:

    def __init__(self, x, y):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def __repr__(self):
        return f"Vector2D({self._x}, {self._y})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector2D):
            return NotImplemented
        return self._x == otro._x and self._y == otro._y

    def __hash__(self):
        return hash((self._x, self._y))

    def __add__(self, otro):
        return Vector2D(self._x + otro._x, self._y + otro._y)

    def __neg__(self):
        ...

## Herencia y `super()`

Una clase puede construirse **a partir de otra**: hereda sus atributos y
métodos, y agrega o cambia solo lo que necesita. `super()` da acceso a la
implementación de la clase madre — típicamente para reusar su `__init__`
en vez de repetirlo.

Una fuerza *es* un vector en el plano, pero además sabe qué interacción
representa ("gravedad", "normal"). Ese **"es un"** es justamente la señal
de que la herencia aplica.

Así está construido SymPy: cada tipo de objeto matemático es una clase que
hereda de otra más general, y por eso todos comparten el mismo
comportamiento básico. Esa jerarquía la abrimos en la semana 12 — hoy
construimos una de dos niveles.

In [ ]:
class Fuerza(Vector2D):

    def __init__(self, x, y, nombre):
        super().__init__(x, y)   # reusa el __init__ de Vector2D
        self._nombre = nombre

    @property
    def nombre(self):
        return self._nombre

    def __repr__(self):
        return f"Fuerza({self._x}, {self._y}, {self._nombre!r})"


peso = Fuerza(0.0, -9.8, "gravedad")

peso, peso.y, isinstance(peso, Vector2D)   # la property y se heredó sin escribirla

## Depuración en vivo: cuando se olvida `super()`

`super().__init__(x, y)` no es decorativo: es lo único que crea `_x` e
`_y`. Si una subclase lo omite, el objeto se construye **a medias** y no
falla en ese momento — falla después, en el primer lugar que use un
atributo que nunca existió. Predice el error antes de ejecutar:

In [ ]:
class FuerzaRota(Vector2D):

    def __init__(self, x, y, nombre):
        # Falta super().__init__(x, y): _x e _y nunca llegan a crearse
        self._nombre = nombre


try:
    FuerzaRota(0.0, -9.8, "gravedad").y
except AttributeError as error:
    print("AttributeError:", error)

Lo incómodo de este error es *dónde* aparece: la línea que falla es la que
lee `.y`, no la del constructor, que es donde está el bug real. Cuando
veas un `AttributeError` sobre un atributo que "deberías tener",
sospecha del `__init__` de la subclase.

## Polimorfismo y *duck typing*

**Polimorfismo**: el mismo código funciona con objetos de clases distintas,
y cada uno responde a su manera. Arriba ya ocurrió — al mostrar `peso`,
Python usó el `__repr__` de `Fuerza`, no el de `Vector2D`.

***Duck typing***: Python no exige que un objeto pertenezca a cierta clase,
solo que sepa hacer lo que se le pide. "Si camina como pato y grazna como
pato, es un pato". La función de abajo suma cualquier cosa que implemente
`__add__`; nunca pregunta de qué tipo es.

In [ ]:
def resultante(vectores):
    total = vectores[0]
    for siguiente in vectores[1:]:
        total = total + siguiente   # lo único que exige es que exista __add__
    return total


resultante([
    Fuerza(0.0, -9.8, "gravedad"),
    Fuerza(0.0, 9.8, "normal"),
    Vector2D(3.0, 0.0),
])

## TODO en clase 2 — laboratorio guiado

Construye `Vector3D` heredando de `Vector2D`. Es el ensayo general de lo
que harán en el Módulo 4 con las clases de SymPy.

1. `__init__(self, x, y, z)` que reuse `super().__init__(x, y)` y guarde
   `self._z`.
2. Una `@property` `z`, de solo lectura como las otras dos.
3. `magnitud`, sobrescrita para incluir la tercera componente.
4. `__repr__`, sobrescrito.
5. `__add__`, sobrescrito.

**El punto 5 es el interesante**, y conviene que primero lo *omitas* para
ver qué pasa: si heredas `__add__` de `Vector2D` sin tocarlo, ¿qué clase
devuelve la suma de dos `Vector3D`? ¿Qué pasa con la componente `z`?

Cuando lo tengas, comprueba que `resultante(...)` —la función de la celda
anterior, que no vamos a modificar— funciona con una lista de `Vector3D`
sin enterarse de que cambió el tipo. Eso es *duck typing* en acción.

In [ ]:
# TODO en clase: completa Vector3D heredando de Vector2D
class Vector3D(Vector2D):

    def __init__(self, x, y, z):
        ...

    @property
    def z(self):
        ...

    @property
    def magnitud(self):
        ...

    def __repr__(self):
        ...

    def __add__(self, otro):
        ...

## Conexión con SymPy (demo motivacional)

Esto es solo una demostración — **no se espera que ustedes escriban código
todavía**. Empezamos con SymPy formalmente en la semana 4.

Todo lo de hoy es, literalmente, cómo está hecho SymPy por dentro: sus
objetos matemáticos son instancias de clases, con métodos especiales, y no
se pueden modificar. Veámoslo.

In [ ]:
import sympy as sp

x = sp.Symbol('x')

x + x   # Symbol define __add__, igual que nuestro Vector2D

In [ ]:
type(x), type(x + x)   # la suma no modifica x: devuelve un objeto nuevo, de otra clase

In [ ]:
# Inmutable, igual que nuestro Vector2D final: no admite atributos nuevos
try:
    x.etiqueta = "posición"
except AttributeError as error:
    print("No se le puede agregar nada:", error)

{x: 3.0}   # y como nunca cambia, es hashable: sirve de llave

Nada de eso es casualidad: `Symbol` es una clase, `x` es una instancia,
`__add__` está definido, y el objeto es inmutable y hashable por las
mismas razones que discutimos hoy. La jerarquía completa de clases sobre
la que está construido SymPy la abrimos en la **semana 12**, cuando les
toque crear sus propios objetos matemáticos.

## Resumen

Hoy vimos por qué un objeto matemático no debe poder modificarse: el bug
del alias, la pérdida del `__hash__` al definir `__eq__`, y el patrón de
atributo privado más `@property` de solo lectura. Después construimos una
jerarquía con `super()`, y comprobamos que el mismo código funciona con
clases distintas gracias al polimorfismo y al *duck typing*.

Es el ensayo general del Módulo 4: en la semana 12 volvemos a esto mismo,
pero heredando de las clases de SymPy en vez de las nuestras.

**Tarea de esta semana:** [`tarea-02.ipynb`](../tarea/tarea-02.ipynb) —
entrega antes de la clase de la Semana 3.

**Próxima clase — Semana 3:** Git y GitHub.